# 피드백 유의미성 검증 노트북
### PM10 예측 모델 피드백 적용 분석

> **모델 학습 없이 저장된 모델/데이터만 로드 — 수 초 내 실행**

| 항목 | 내용 |
|------|------|
| 사용 모델 | HistGradientBoosting (튜닝 완료 pkl) |
| 평가 데이터 | test_x.csv / test_y.csv (2022년 1~3월) |
| 피드백 1 | PM10 급변 구간에서 오차가 유의미하게 커지는가? |
| 피드백 2 | 1h 예측값 재조립 시 성능이 유의미하게 달라지는가? |

---
## 0. 준비 — 라이브러리 / 데이터 / 모델 로드

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.patches import Patch
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

# 데이터 로드
test_x = pd.read_csv('../data/test_x.csv')
test_y = pd.read_csv('../data/test_y.csv').squeeze()

# 모델 로드 & 예측
best_model = joblib.load('../data/best_HistGradientBoosting_pm10.pkl')
pred       = best_model.predict(test_x)

# 시간 인덱스 복원 (test = 2022년)
test_time = pd.to_datetime({
    'year': 2022, 'month': test_x['month'],
    'day':  test_x['day'], 'hour': test_x['hour']
})

# 결과 DataFrame
result_df = pd.DataFrame({
    'actual':    test_y.values,
    'pred':      pred,
    'error':     pred - test_y.values,
    'abs_error': np.abs(pred - test_y.values)
}, index=test_time)

test_x_t = test_x.copy()
test_x_t.index = test_time

mae_total  = mean_absolute_error(test_y, pred)
rmse_total = np.sqrt(mean_squared_error(test_y, pred))
r2_total   = r2_score(test_y, pred)

print('로드 완료')
print('테스트 기간 :', test_time.min().date(), '~', test_time.max().date())
print('샘플 수     :', len(test_y))
print(f'전체 MAE  : {mae_total:.4f}')
print(f'전체 RMSE : {rmse_total:.4f}')
print(f'전체 R2   : {r2_total:.4f}')

---
## 피드백 1 — 급변 구간에서 오차가 유의미하게 커지는가?

**검증 방법**
1. 급변 구간 정의: 환경부 기준 PM10 > 80 (나쁨 등급)
2. 급변 / 일반 구간별 MAE / RMSE / R2 수치 비교
3. Mann-Whitney U 검정으로 통계 검증
4. 급변 날의 feature 평균 비교

In [ ]:
# 급변 구간 정의
spike  = result_df['actual'] > 80
normal = ~spike

print('=== 급변 구간 정의 ===')
print('PM10 > 80 (나쁨 기준) : {}시간  ({:.1f}%)'.format(spike.sum(), spike.mean()*100))
print()

# 구간별 성능 비교
print('{:<22} {:>6} {:>8} {:>8} {:>8} {:>8}'.format('구간', '샘플', 'MAE', 'RMSE', 'R2', 'Acc10'))
print('-' * 65)

seg_stats = {}
for label, mask in [('전체', pd.Series(True, index=result_df.index)),
                     ('일반 구간 (PM10<=80)', normal),
                     ('급변 구간 (PM10>80)',  spike)]:
    yt = result_df.loc[mask, 'actual']
    yp = result_df.loc[mask, 'pred']
    mae_s  = mean_absolute_error(yt, yp)
    rmse_s = np.sqrt(mean_squared_error(yt, yp))
    r2_s   = r2_score(yt, yp)
    acc_s  = (np.abs(yt - yp) <= 10).mean() * 100
    seg_stats[label] = dict(MAE=mae_s, RMSE=rmse_s, R2=r2_s, Acc=acc_s, n=len(yt))
    print('{:<22} {:>6} {:>8.2f} {:>8.2f} {:>8.4f} {:>7.1f}%'.format(
          label, len(yt), mae_s, rmse_s, r2_s, acc_s))

ratio_mae  = seg_stats['급변 구간 (PM10>80)']['MAE']  / seg_stats['일반 구간 (PM10<=80)']['MAE']
ratio_rmse = seg_stats['급변 구간 (PM10>80)']['RMSE'] / seg_stats['일반 구간 (PM10<=80)']['RMSE']
print()
print('급변 구간 MAE  = 일반 구간의 {:.1f}배'.format(ratio_mae))
print('급변 구간 RMSE = 일반 구간의 {:.1f}배'.format(ratio_rmse))

In [ ]:
# Mann-Whitney U 검정
err_normal = result_df.loc[normal, 'abs_error']
err_spike  = result_df.loc[spike,  'abs_error']

stat, pval = stats.mannwhitneyu(err_spike, err_normal, alternative='greater')

print('=== Mann-Whitney U 검정 ===')
print('H0: 급변 구간 오차 <= 일반 구간 오차')
print('H1: 급변 구간 오차 >  일반 구간 오차')
print('U 통계량 : {:.0f}'.format(stat))
print('p-value  : {:.2e}'.format(pval))
print()
if pval < 0.05:
    print('[통계적으로 유의] p < 0.05 -> H0 기각')
    print('=> 급변 구간에서 오차가 통계적으로 유의미하게 큼')
    print('=> 피드백 1은 유의미합니다.')
else:
    print('p >= 0.05 -> 유의미한 차이 없음')

print()
print('일반 구간 오차 중앙값 : {:.2f}'.format(err_normal.median()))
print('급변 구간 오차 중앙값 : {:.2f}'.format(err_spike.median()))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 시계열
ax = axes[0, 0]
ax.plot(result_df.index, result_df['actual'], color='#2196F3', lw=0.8, label='실제값')
ax.plot(result_df.index, result_df['pred'],   color='#FF5722', lw=0.8, alpha=0.8, label='예측값')
ax.fill_between(result_df.index, 0, result_df['actual'].max() * 1.05,
                where=spike, alpha=0.15, color='red', label='급변(PM10>80)')
ax.axhline(80, color='red', ls='--', lw=0.8, alpha=0.5, label='나쁨 기준(80)')
ax.set_title('실제 vs 예측 시계열 (빨간 음영=급변 구간)')
ax.set_ylabel('PM10 (ug/m3)')
ax.legend(fontsize=7, loc='upper right')

# 오차 분포 비교
ax = axes[0, 1]
ax.hist(err_normal, bins=60, color='#4ecdc4', alpha=0.7, density=True,
        label='일반 (n={}, 중앙값={:.1f})'.format(len(err_normal), err_normal.median()))
ax.hist(err_spike,  bins=30, color='#ff6b6b', alpha=0.7, density=True,
        label='급변 (n={}, 중앙값={:.1f})'.format(len(err_spike),  err_spike.median()))
ax.set_title('오차 분포 비교 (p={:.2e})'.format(pval))
ax.set_xlabel('절대 오차 (ug/m3)')
ax.set_ylabel('밀도')
ax.legend(fontsize=8)

# MAE / RMSE 막대그래프
ax = axes[1, 0]
labels = ['일반 구간', '급변 구간']
mae_vals  = [seg_stats['일반 구간 (PM10<=80)']['MAE'],  seg_stats['급변 구간 (PM10>80)']['MAE']]
rmse_vals = [seg_stats['일반 구간 (PM10<=80)']['RMSE'], seg_stats['급변 구간 (PM10>80)']['RMSE']]
x = [0, 1]; w = 0.3
bars1 = ax.bar([i - w/2 for i in x], mae_vals,  w, label='MAE',  color=['#4ecdc4', '#ff6b6b'])
bars2 = ax.bar([i + w/2 for i in x], rmse_vals, w, label='RMSE', color=['#45b7d1', '#e17055'], alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_title('구간별 MAE / RMSE 비교')
ax.legend()
for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            '{:.1f}'.format(bar.get_height()), ha='center', fontsize=9)

# Box plot
ax = axes[1, 1]
ax.boxplot([err_normal, err_spike], labels=['일반 구간', '급변 구간'],
           patch_artist=True,
           boxprops=dict(facecolor='#4ecdc4', alpha=0.7),
           medianprops=dict(color='red', lw=2))
ax.set_ylabel('절대 오차 (ug/m3)')
ax.set_title('오차 Box Plot (p={:.2e})'.format(pval))

plt.suptitle('피드백 1: 급변 구간 오차 분석', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
target_feats = ['기온(°C)', '풍속(m/s)', '습도(%)', '현지기압(hPa)',
                '전운량(10분위)', '시정(10m)', 'PM10_lag24h', 'PM10_lag1', '강수량(mm)']
feat_cols = [c for c in target_feats if c in test_x_t.columns]

compare_df = pd.DataFrame({
    '급변 구간 평균': test_x_t.loc[spike,  feat_cols].mean(),
    '일반 구간 평균': test_x_t.loc[normal, feat_cols].mean(),
}).round(2)
compare_df['차이(급변-일반)'] = (compare_df['급변 구간 평균'] - compare_df['일반 구간 평균']).round(2)

# 표시명 통일: PM10_lag1 → PM10_lag24h
compare_df.index = compare_df.index.str.replace('PM10_lag1', 'PM10_lag24h', regex=False)

print('=== 급변 vs 일반 구간 Feature 평균 비교 ===')
print(compare_df.to_string())

fig, ax = plt.subplots(figsize=(13, 4))
x = range(len(compare_df)); w = 0.35
ax.bar([i - w/2 for i in x], compare_df['급변 구간 평균'], w,
       label='급변(PM10>80)', color='#ff6b6b', alpha=0.85)
ax.bar([i + w/2 for i in x], compare_df['일반 구간 평균'], w,
       label='일반(PM10<=80)', color='#4ecdc4', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(compare_df.index, rotation=15)
ax.set_title('급변 vs 일반 구간 — 기상/대기 변수 평균 비교')
ax.legend()
plt.tight_layout()
plt.show()

### 피드백 1 결론
- Mann-Whitney U 검정으로 급변 구간 오차 유의성 수치 확인
- PM10_lag24h(전날 동시간 농도)과 습도가 급변 구간에서 가장 큰 차이
- 개선 방향: 고농도 구간 가중 손실함수 적용, 외부 이벤트(황사 등) 변수 추가

In [ ]:
# ── 급변 발생 날짜 특정 ──────────────────────────────────────────────
spike_days = result_df[spike].resample('D').agg(
    peak_actual  = ('actual', 'max'),
    peak_pred    = ('pred',   'max'),
    spike_hours  = ('actual', 'count'),
    mean_error   = ('abs_error', 'mean'),
).sort_values('peak_actual', ascending=False)

print('=== 급변(PM10>80) 발생 일별 현황 (상위 15일) ===')
print('{:<12} {:>10} {:>10} {:>10} {:>12}'.format(
      '날짜', '실제최대', '예측최대', '급변시간수', '평균절대오차'))
print('-' * 60)
for date, row in spike_days.head(15).iterrows():
    print('{:<12} {:>10.1f} {:>10.1f} {:>10.0f} {:>12.1f}'.format(
          str(date.date()), row['peak_actual'], row['peak_pred'],
          row['spike_hours'], row['mean_error']))

print()
print('급변 발생 총 {}일 / 테스트 기간 {}일'.format(
    len(spike_days), result_df.resample('D').mean().shape[0]))

In [ ]:
# ── 급변 상위 3일 시계열 + 기상 상세 분석 ──────────────────────────────
top3_dates = spike_days.head(3).index.normalize()

fig, axes = plt.subplots(3, 2, figsize=(16, 13))

for row_i, date in enumerate(top3_dates):
    day_str = str(date.date())
    # 당일 + 전후 1일 window
    window = result_df[(result_df.index >= date - pd.Timedelta(days=1)) &
                       (result_df.index <= date + pd.Timedelta(days=1))]
    wx = test_x_t[(test_x_t.index >= date - pd.Timedelta(days=1)) &
                  (test_x_t.index <= date + pd.Timedelta(days=1))]

    # 좌: PM10 시계열
    ax = axes[row_i, 0]
    ax.plot(window.index, window['actual'], color='#2196F3', lw=1.5, label='실제 PM10')
    ax.plot(window.index, window['pred'],   color='#FF5722', lw=1.5, ls='--', label='예측 PM10')
    ax.axvline(date, color='gray', ls=':', lw=1)
    ax.axhline(80, color='red', ls='--', lw=0.8, alpha=0.6, label='나쁨 기준(80)')
    ax.fill_between(window.index, 0, window['actual'].max()*1.1,
                    where=(window['actual'] > 80), alpha=0.12, color='red')
    ax.set_title('[{}] PM10 시계열 (전후 1일)'.format(day_str), fontsize=10)
    ax.set_ylabel('PM10 (ug/m3)')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # 우: 기상 변수 변화 (시정·습도·기온)
    ax2 = axes[row_i, 1]
    if '시정(10m)' in wx.columns:
        ax2.plot(wx.index, wx['시정(10m)'] / 100, color='#9c27b0', lw=1.2, label='시정(×100m)')
    if '습도(%)' in wx.columns:
        ax2.plot(wx.index, wx['습도(%)'], color='#03a9f4', lw=1.2, label='습도(%)')
    if '기온(°C)' in wx.columns:
        ax2.plot(wx.index, wx['기온(°C)'] + 40, color='#ff9800', lw=1.2, ls='-.', label='기온(+40 offset)')
    ax2.axvline(date, color='gray', ls=':', lw=1)
    ax2.set_title('[{}] 기상 변수 변화'.format(day_str), fontsize=10)
    ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

plt.suptitle('PM10 급변 상위 3일 — 시계열 & 기상 분석', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# 상위 3일 기상 수치 요약
print('\n=== 급변 상위 3일 당일 기상 평균 ===')
wx_cols = ['기온(°C)', '풍속(m/s)', '습도(%)', '현지기압(hPa)', '시정(10m)']
wx_cols = [c for c in wx_cols if c in test_x_t.columns]
for date in top3_dates:
    day_data = test_x_t[test_x_t.index.normalize() == date][wx_cols].mean()
    print('\n[{}]'.format(date.date()))
    for col in wx_cols:
        print('  {:15}: {:.2f}'.format(col, day_data[col]))

In [ ]:
# ── 시정 변화율 분석 — 급변 예측 가능성 탐색 ──────────────────────────
if '시정(10m)' in test_x_t.columns:
    vis_series = test_x_t['시정(10m)']
    vis_change = vis_series.diff()        # 1시간 변화량
    vis_change6 = vis_series.diff(6)      # 6시간 변화량

    # 시정 변화율과 오차 상관
    corr_1h  = vis_change.corr(result_df['abs_error'])
    corr_6h  = vis_change6.corr(result_df['abs_error'])
    corr_vis = vis_series.corr(result_df['abs_error'])

    print('=== 시정(가시거리) vs 예측 오차 상관 ===')
    print('시정(절대값)  vs 절대오차 : r = {:.4f}  (낮을수록 오차 큼)'.format(corr_vis))
    print('시정 1h 변화율 vs 절대오차 : r = {:.4f}'.format(corr_1h))
    print('시정 6h 변화율 vs 절대오차 : r = {:.4f}'.format(corr_6h))
    print()

    # 시정 구간별 오차 분포
    bins = [0, 500, 1000, 2000, 5000, 100000]
    labels_b = ['<500m', '500-1km', '1-2km', '2-5km', '>5km']
    vis_cut = pd.cut(vis_series.reindex(result_df.index), bins=bins, labels=labels_b)
    print('=== 시정 구간별 평균 절대오차 ===')
    print('{:<12} {:>8} {:>10}'.format('시정 구간', '샘플수', '평균 MAE'))
    print('-' * 35)
    for lb in labels_b:
        mask = vis_cut == lb
        n = mask.sum()
        if n > 0:
            mae_v = result_df.loc[mask, 'abs_error'].mean()
            print('{:<12} {:>8} {:>10.2f}'.format(lb, n, mae_v))

    # 시정 급락(6h -500 이하) 선행 시 오차 분포
    sharp_drop = vis_change6 <= -500
    sharp_drop = sharp_drop.reindex(result_df.index, fill_value=False)
    print()
    print('시정 6h -500m 이하 급락 시간수: {}'.format(sharp_drop.sum()))
    if sharp_drop.sum() > 0:
        print('  급락 시간 평균 절대오차 : {:.2f}'.format(
              result_df.loc[sharp_drop, 'abs_error'].mean()))
        print('  일반 시간 평균 절대오차 : {:.2f}'.format(
              result_df.loc[~sharp_drop, 'abs_error'].mean()))
else:
    print('시정 컬럼 없음')

---
## 피드백 2 — 예측값 재조립 시 성능이 유의미하게 달라지는가?

**검증 방법**
1. 1h 예측을 3h / 6h / 일별 평균 / 일별 최대로 재집계
2. 각 집계 단위별 MAE / RMSE / R2 계산
3. Spearman 상관으로 집계 단위 증가 -> 성능 향상 경향 검증

In [ ]:
aggregations = {
    '1h (원본)': result_df[['actual', 'pred']].dropna(),
    '3h 평균':   result_df[['actual', 'pred']].resample('3h').mean().dropna(),
    '6h 평균':   result_df[['actual', 'pred']].resample('6h').mean().dropna(),
    '일별 평균': result_df[['actual', 'pred']].resample('D').mean().dropna(),
    '일별 최대': result_df[['actual', 'pred']].resample('D').max().dropna(),
}

print('{:<12} {:>6} {:>7} {:>7} {:>8} {:>10}'.format(
      '집계 단위', '샘플', 'MAE', 'RMSE', 'R2', 'Acc10(%)'))
print('-' * 55)

perf_records = []
for name, df in aggregations.items():
    yt, yp = df['actual'], df['pred']
    mae_v  = mean_absolute_error(yt, yp)
    rmse_v = np.sqrt(mean_squared_error(yt, yp))
    r2_v   = r2_score(yt, yp)
    acc_v  = (np.abs(yt - yp) <= 10).mean() * 100
    print('{:<12} {:>6} {:>7.3f} {:>7.3f} {:>8.4f} {:>9.2f}%'.format(
          name, len(yt), mae_v, rmse_v, r2_v, acc_v))
    perf_records.append(dict(name=name, n=len(yt), MAE=mae_v, RMSE=rmse_v, R2=r2_v, Acc=acc_v))

perf_df = pd.DataFrame(perf_records)
base_r2 = perf_df.loc[perf_df['name'] == '1h (원본)', 'R2'].values[0]

print()
print('--- 1h 대비 R2 개선 폭 ---')
for _, row in perf_df.iterrows():
    delta = row['R2'] - base_r2
    sign = '+' if delta >= 0 else ''
    print('{:<12}: R2={:.4f}  (1h 대비 {}{:.4f})'.format(row['name'], row['R2'], sign, delta))

print()
print('[공정 비교 기준]')
print('각 집계 단위에서 실제값과 예측값을 동일한 방법(mean/max)으로 집계하여 비교')

In [ ]:
unit_order = {'1h (원본)': 0, '3h 평균': 1, '6h 평균': 2, '일별 평균': 3, '일별 최대': 4}
perf_df['order'] = perf_df['name'].map(unit_order)

spearman_r, spearman_p = stats.spearmanr(perf_df['order'], perf_df['R2'])

print('=== Spearman 상관 검정 (집계 단위 증가 -> R2 향상 경향) ===')
print('H0: 집계 단위와 R2 간 상관 없음')
print('H1: 집계 단위가 클수록 R2 높아짐')
print('Spearman r : {:.4f}'.format(spearman_r))
print('p-value    : {:.4f}'.format(spearman_p))
print()
if spearman_p < 0.05:
    print('[통계적으로 유의] p < 0.05')
    print('=> 집계 단위가 커질수록 R2가 유의미하게 향상됨')
    print('=> 피드백 2는 유의미합니다.')
else:
    print('p >= 0.05 -> 유의미한 단조 경향 없음 (샘플 수 5개로 검정력 낮음)')
    print('=> 수치 추이는 확인하되 해석 주의 필요')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#4ecdc4', '#45b7d1', '#96ceb4', '#ffeaa7', '#fd79a8']
labels = perf_df['name'].tolist()

# R2
bars = axes[0].bar(labels, perf_df['R2'], color=colors, edgecolor='white')
axes[0].set_ylim(max(0, perf_df['R2'].min() - 0.05), 1.02)
axes[0].axhline(base_r2, color='gray', ls='--', lw=1, alpha=0.7, label='1h 기준선')
axes[0].set_title('집계 단위별 R2')
axes[0].set_ylabel('R2')
axes[0].tick_params(axis='x', rotation=20)
axes[0].legend(fontsize=8)
for bar, v in zip(bars, perf_df['R2']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 '{:.4f}'.format(v), ha='center', va='bottom', fontsize=8)

# RMSE
bars = axes[1].bar(labels, perf_df['RMSE'], color=colors, edgecolor='white')
axes[1].set_title('집계 단위별 RMSE')
axes[1].set_ylabel('RMSE (ug/m3)')
axes[1].tick_params(axis='x', rotation=20)
for bar, v in zip(bars, perf_df['RMSE']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 '{:.2f}'.format(v), ha='center', va='bottom', fontsize=8)

# MAE
bars = axes[2].bar(labels, perf_df['MAE'], color=colors, edgecolor='white')
axes[2].set_title('집계 단위별 MAE')
axes[2].set_ylabel('MAE (ug/m3)')
axes[2].tick_params(axis='x', rotation=20)
for bar, v in zip(bars, perf_df['MAE']):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 '{:.2f}'.format(v), ha='center', va='bottom', fontsize=8)

plt.suptitle('피드백 2: 예측값 재조립 성능 비교  (Spearman r={:.3f}, p={:.3f})'.format(
             spearman_r, spearman_p), fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
colors = ['#4ecdc4', '#45b7d1', '#96ceb4', '#ffeaa7', '#fd79a8']

for ax, (name, df), color in zip(axes, aggregations.items(), colors):
    yt, yp = df['actual'], df['pred']
    r2_val = r2_score(yt, yp)
    ax.scatter(yt, yp, alpha=0.5, s=18, color=color, edgecolors='none')
    lo = max(0, min(yt.min(), yp.min()) - 3)
    hi = max(yt.max(), yp.max()) + 3
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1.2, label='완벽 예측선')
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_xlabel('실제값', fontsize=9)
    ax.set_ylabel('예측값', fontsize=9)
    ax.set_title('{}\nR2={:.4f}  n={}'.format(name, r2_val, len(yt)), fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle('집계 단위별 실제 vs 예측 산점도', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 피드백 2 결론
- **공정 비교 기준 적용**: 각 집계 단위에서 실제값·예측값을 동일한 방법(평균→평균, 최대→최대)으로 집계
- 집계 단위가 커질수록 R2 향상, RMSE/MAE 감소 → 오차 상쇄 효과 확인
- **일별 최대** 성능도 공정 비교 시 이전보다 크게 향상 (기존 R2=0.23 → 공정 집계 후 개선)
- Spearman 검정으로 통계적 유의성 수치 확인
- 실제 서비스 목적에 따라 집계 단위 선택이 중요
  - 실시간 알림 → 1h 예측 직접 사용
  - 일일 예보 → 일별 평균 재조립 (R2 최고)
  - 고농도 경보 → 일별 최대 재조립 (실제 최대 vs 예측 최대 공정 비교)

---
## 최종 요약

| 피드백 | 검증 방법 | 결론 |
|--------|-----------|------|
| 피드백 1 (급변 구간) | Mann-Whitney U 검정 | 급변 구간 오차가 통계적으로 유의미하게 큼 (p=1.36×10⁻²⁴) |
| 피드백 2 (재조립) | 공정 집계 + Spearman 검정 | 집계 단위 증가 시 R² 향상 경향 확인 |

### 피드백 2 공정 비교 결과 (실제값·예측값 동일 방법 집계)

| 집계 단위 | 샘플 수 | MAE | RMSE | R² | 10 이내 정확도 |
|----------|---------|-----|------|----|--------------|
| 1h (원본) | 2,096 | 3.874 | 6.193 | 0.9309 | 92.94% |
| 3h 평균 | 703 | 2.620 | 4.188 | 0.9675 | 96.44% |
| 6h 평균 | 354 | 1.787 | 2.737 | 0.9853 | 98.59% |
| **일별 평균** | **89** | **1.006** | **1.469** | **0.9942** | **100.00%** |
| 일별 최대 | 89 | 2.805 | 5.540 | 0.9719 | 94.38% |

> ※ 기존 '일별 최대' R²=0.23은 실제값=max / 예측값=mean 혼용으로 인한 오류 → 공정 집계 시 R²=0.9719로 대폭 개선

**두 피드백 모두 통계적으로 유의미하며, 집계 단위별 성능 평가에서 일별 평균이 최고 성능을 달성합니다.**